# Search

Run from the repo root, after `python -m similarity.preprocessing`.

In [ ]:
import time
import numpy as np
import pandas as pd
import similarity.search as s
from similarity.preprocessing import load_products

products = load_products()
f = s.load_features()
s.WEIGHTS

## one product

In [ ]:
row = 3
products.loc[row, ["name", "brand", "child_category", "price", "colours", "weight_g"]]

In [ ]:
order, score = s.rank(f, row)
products.loc[order[:10], ["name", "child_category", "price", "rating_adjusted"]].assign(score=score[:10].round(3))

In [ ]:
s.find_similar_products(products.loc[row, "uniq_id"], 5)

## breakdown

In [ ]:
s.score_breakdown(f, row, order[0]).round(3)

In [ ]:
s.score_breakdown(f, row, 1).round(3)

In [ ]:
scores = s.attribute_scores(f, row)
rows = [1, 7, 2]
pd.DataFrame({
    "name": products.loc[rows, "name"].str[:45],
    "text_only": scores["text"][0][rows].round(3),
    "combined": s.combine(scores, s.WEIGHTS)[rows].round(3),
})

## missing attributes

In [ ]:
has = pd.DataFrame({
    "text": products["name"].str.len() > 0,
    "category": (f.category != -1) | (f.browsenode != -1),
    "gender": f.gender != -1,
    "price": ~np.isnan(f.price),
    "brand": f.brand != -1,
    "colour": f.colours.getnnz(axis=1) > 0,
    "weight": ~np.isnan(f.weight),
})
has.mean().round(3)

In [ ]:
has.sum(axis=1).value_counts().sort_index()

In [ ]:
few = has.index[has.sum(axis=1) == 2][0]
has.loc[few]

In [ ]:
order, score = s.rank(f, few)
pd.concat([
    products.loc[[few], ["name", "child_category", "price"]],
    products.loc[order[:5], ["name", "child_category", "price"]].assign(score=score[:5].round(3)),
])

In [ ]:
s.score_breakdown(f, few, order[0]).round(3)

## duplicates

In [ ]:
asin = products["asin"].value_counts().index[0]
twins = products.index[products["asin"] == asin]
products.loc[twins, ["uniq_id", "asin", "name", "price"]]

In [ ]:
order, score = s.rank(f, twins[0])
products.loc[order[:5], ["uniq_id", "asin", "name"]].assign(score=score[:5].round(3))

In [ ]:
(products.loc[order, "asin"] == asin).sum(), products.loc[order, "asin"].is_unique, len(order)

## ties

In [ ]:
order, score = s.rank(f, row)
top = products.loc[order[:20], ["name", "rating_adjusted"]].assign(score=score[:20].round(3))
top[top["score"].duplicated(keep=False)]

## random products

In [ ]:
r = products.sample(1, random_state=1).index[0]
order, score = s.rank(f, r)
pd.concat([
    products.loc[[r], ["name", "child_category", "price"]],
    products.loc[order[:5], ["name", "child_category", "price"]].assign(score=score[:5].round(3)),
])

In [ ]:
r = products.sample(1, random_state=2).index[0]
order, score = s.rank(f, r)
pd.concat([
    products.loc[[r], ["name", "child_category", "price"]],
    products.loc[order[:5], ["name", "child_category", "price"]].assign(score=score[:5].round(3)),
])

## no shared words

In [ ]:
order, score = s.rank(f, 0)
text = s.text_scores(f, 0)[0]
products.loc[order[:8], ["name", "child_category", "price"]].assign(text=text[order[:8]].round(2), score=score[:8].round(3))

In [ ]:
closest = np.argsort(-text)[1:6]
position = pd.Series(np.arange(len(order)), index=order)
products.loc[closest, ["name", "child_category", "price"]].assign(text=text[closest].round(2), rank=position.reindex(closest).values)

## text only vs combined

In [ ]:
rows = np.random.default_rng(0).choice(len(f.ids), 500, replace=False)
text_only = s.result_checks(f, rows, {"text": 1.0})
combined = s.result_checks(f, rows)
pd.DataFrame({"text_only": text_only.mean(), "combined": combined.mean()}).round(3)

## timing

In [ ]:
start = time.perf_counter()
for r in range(100):
    s.rank(f, r)
(time.perf_counter() - start) / 100 * 1000